### Set up evironment

In [1]:
%pip install accelerate bitsandbytes google-cloud-aiplatform peft trl scikit-learn tokenizers torch transformers unsloth evaluate python-dotenv sentencepiece protobuf matplotlib

Note: you may need to restart the kernel to use updated packages.


### import library

In [24]:
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
from datasets import load_dataset
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from huggingface_hub import login
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from accelerate import Accelerator

### Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Login huggingface

In [4]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub successfully.")
else:
    print("Failed to retrieve Hugging Face token.")

Logged in to Hugging Face Hub successfully.


### Load dataset

In [5]:
print("Loading dataset...")
dataset = load_dataset("stanfordnlp/imdb")
print("Dataset loaded successfully.")
dataset

Loading dataset...
Dataset loaded successfully.


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

### Visualize training data

In [6]:
print("Positive reviews:", len(dataset['train'].filter(lambda x: x['label'] == 1)))
print("Negative reviews:", len(dataset['train'].filter(lambda x: x['label'] == 0)))

Positive reviews: 12500
Negative reviews: 12500


### Tokenization and preprocessing

In [7]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

def tokenizer_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

print("Starting tokenization...")
tokenized_datasets = dataset.map(tokenizer_function, batched=True)
print("Tokenization completed.")
tokenized_datasets

Starting tokenization...
Tokenization completed.


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

### Validate

In [8]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

### Select model

In [ ]:
sys.path.append(os.path.abspath("../"))
from transformer.model import TransformerClassifier

vocab_size = len(tokenizer)

config = {
    "vocab_size": vocab_size,
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 2,
    "d_ff": 512,
    "n_classes": 2,
    "max_len": 512,
    "dropout": 0.1
}

model = TransformerClassifier(**config).to(device)

print(f"Model dã sẵn sàng với {vocab_size} tokens trong từ điển.")


Model dã sẵn sàng với 128001 tokens trong từ điển.


In [27]:
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8)
eval_dataloader = DataLoader(tokenized_datasets["test"], batch_size=8)

### Manual training loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = torch.nn.CrossEntropyLoss()
model = TransformerClassifier(**config).to(device)

num_epochs = 3
print(f"Training model for {num_epochs} epochs...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in progress_bar:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits
        loss = loss_fn(logits, labels) 
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    model.eval()
    correct = 0
    total = 0
    eval_progress = tqdm(eval_dataloader, desc=f"Epoch {epoch+1} [Eval]")
    
    with torch.no_grad():
        for batch in eval_progress:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)
    print(f"--- Results Epoch {epoch+1} ---")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Accuracy on test set: {accuracy:.2f}%")
    print("-" * 30)

Training model for 3 epochs...


Epoch 1 [Eval]: 100%|██████████| 3125/3125 [00:21<00:00, 147.35it/s]


--- Results Epoch 1 ---
Average Loss: 0.5845
Accuracy on test set: 76.64%
------------------------------


Epoch 2 [Eval]: 100%|██████████| 3125/3125 [00:24<00:00, 128.42it/s]


--- Results Epoch 2 ---
Average Loss: 0.4525
Accuracy on test set: 79.73%
------------------------------


Epoch 3 [Eval]: 100%|██████████| 3125/3125 [00:21<00:00, 147.74it/s]

--- Results Epoch 3 ---
Average Loss: 0.4092
Accuracy on test set: 81.41%
------------------------------


### Accelerate

In [28]:
model = TransformerClassifier(**config).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = torch.nn.CrossEntropyLoss()

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)


num_epochs = 3
print(f"Accelerate training model for {num_epochs} epochs...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in progress_bar:

        input_ids = batch['input_ids'].to(accelerator.device)
        attention_mask = batch['attention_mask'].to(accelerator.device)
        labels = batch['label'].to(accelerator.device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits
        loss = loss_fn(logits, labels) 
        
        accelerator.backward(loss)
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    model.eval()
    correct = 0
    total = 0
    eval_progress = tqdm(eval_dataloader, desc=f"Epoch {epoch+1} [Eval]")
    
    with torch.no_grad():
        for batch in eval_progress:
            input_ids = batch['input_ids'].to(accelerator.device)
            attention_mask = batch['attention_mask'].to(accelerator.device)
            labels = batch['label'].to(accelerator.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)
    print(f"--- Results Epoch {epoch+1} ---")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Accuracy on test set: {accuracy:.2f}%")
    print("-" * 30)
    

Accelerate training model for 3 epochs...


Epoch 1 [Eval]: 100%|██████████| 3125/3125 [00:19<00:00, 158.36it/s]


--- Results Epoch 1 ---
Average Loss: 0.5814
Accuracy on test set: 75.75%
------------------------------


Epoch 2 [Eval]: 100%|██████████| 3125/3125 [00:21<00:00, 145.94it/s]


--- Results Epoch 2 ---
Average Loss: 0.4719
Accuracy on test set: 78.62%
------------------------------


Epoch 3 [Eval]: 100%|██████████| 3125/3125 [00:21<00:00, 148.23it/s]

--- Results Epoch 3 ---
Average Loss: 0.4286
Accuracy on test set: 80.85%
------------------------------


### Trainer API

In [13]:
model = TransformerClassifier(**config)
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./transformer",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=500,               
        load_best_model_at_end=True,
        metric_for_best_model="accuracy" 
    ),
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.534500,0.527848,0.743200,0.742607
2,0.472700,0.496538,0.770160,0.768438
3,0.428800,0.474716,0.787600,0.787552


TrainOutput(global_step=9375, training_loss=0.5094154085286459, metrics={'train_runtime': 412.2668, 'train_samples_per_second': 181.921, 'train_steps_per_second': 22.74, 'total_flos': 0.0, 'train_loss': 0.5094154085286459, 'epoch': 3.0})